# Datalog

A self-contained refresher on **Datalog** — a declarative query language and deductive-database formalism: a decidable subset of Prolog built for *recursive queries over relational data*.

**Domain:** Symbolic AI & Logic  ·  **runnable:** yes (a ~40-line pure-Python engine, stdlib only)

## 1. What & Why

**Datalog** is a rule-based query language. You declare a set of **facts** (an *extensional* database, the EDB) and **rules** that derive new facts (the *intensional* database, the IDB), then ask **queries**. It looks like Prolog without function symbols:

```
parent(tom, bob).                          % a fact
ancestor(X, Y) :- parent(X, Y).            % a rule (base case)
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).   % recursive rule
```

**The problem it solves.** SQL is great until you need *recursion* — "find everyone reachable in a graph," "all parts that transitively contain part X," "all roles a user inherits." In plain SQL that means awkward `WITH RECURSIVE` CTEs; in application code it means hand-rolled fixpoint loops. Datalog makes recursive, relational inference a *first-class, declarative primitive*: you state the rule and the engine computes the least fixpoint for you, with a **guarantee that it terminates**.

**Reach for it when:**
- You have relational/graph data and need **recursive** queries (reachability, transitive closure, ancestry, dependency graphs, inheritance).
- You want a **deductive database** or rules engine whose queries are guaranteed decidable and terminating.
- You're doing **program analysis** (points-to, dataflow), **network/access-control policy**, or **knowledge-graph** reasoning — all huge Datalog application areas (Soufflé, LogicBlox, Datomic, RDFox).
- You want SQL-like declarativeness *plus* recursion and a clean fixpoint semantics.

**Skip it when:** the work is non-relational, needs arbitrary computation / data structures (use Prolog or a general language), is purely non-recursive (plain SQL is fine), or is numeric/ML-heavy. Datalog deliberately trades expressive power for decidability.

## 2. Mental Model

> **Datalog = SQL views that are allowed to reference themselves, evaluated bottom-up to a fixpoint.**

A rule is a *view definition*; the recursive rule says "this view is defined partly in terms of itself." Evaluation is **bottom-up, set-at-a-time**: start from the EDB facts, fire every rule that can fire, add whatever new facts you derive, and repeat. Because there are **no function symbols** the universe of possible facts is *finite* (only the constants already present), so this loop must reach a **least fixpoint** and stop.

```
EDB facts ─┐
           ▼
  ┌──────────────────────┐   derive new facts
  │  apply ALL rules once │ ───────────────────►  add to fact set
  └──────────────────────┘                              │
           ▲                                             │
           └──────────  any new facts? yes → repeat  ────┘
                        no new facts? → FIXPOINT, done
```

Contrast with **Prolog**, which is *top-down* (start from the query goal, search depth-first with backtracking). That top-down search can loop forever on left-recursive rules like `ancestor(X,Y) :- ancestor(X,Z), parent(Z,Y)`. The *same* rules in Datalog's bottom-up engine simply terminate. Bottom-up + finite Herbrand universe = the whole reason Datalog is decidable.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Fact / EDB** | A ground (variable-free) atom you assert: `parent(tom, bob).` The *extensional* database — your raw data. |
| **Rule** | `Head :- Body₁, …, Bodyₙ.` — "Head holds if all body atoms hold." A Horn clause with no function symbols. |
| **IDB** | The *intensional* database: predicates defined by rules (e.g. `ancestor`), computed from the EDB. |
| **Atom / Predicate** | `parent(X, Y)` is an atom of predicate `parent/2`. Lowercase = constant, Uppercase = variable. |
| **Recursion** | A predicate defined in terms of itself — Datalog's headline feature vs SQL. |
| **Least fixpoint** | The unique smallest set of facts closed under all rules; what bottom-up evaluation computes. |
| **Naive evaluation** | Re-derive *all* facts every round until nothing new appears. Simple, correct, redundant. |
| **Semi-naive evaluation** | Optimization: each round only join against facts *newly* derived last round. Same answer, far less work. |
| **Safety / range restriction** | Every variable in the head (and in negated/comparison atoms) must appear in a positive body atom — guarantees finite, well-defined results. |
| **Stratified negation** | Negation (`not`) is allowed only if predicates can be layered so nothing depends recursively on its own negation — keeps semantics unambiguous. |
| **Set semantics** | Results are *sets* of facts — no duplicates, order-independent (unlike Prolog's ordered, backtracking answers). |

## 4. Setup

Datalog isn't one product; it's a formalism with many engines. Real-world options:

- **[pyDatalog](https://sites.google.com/site/pydatalog/)** — pip-installable Datalog embedded in Python (`pip install pyDatalog`). Easiest on-ramp, though unmaintained on newer Pythons.
- **[Soufflé](https://souffle-lang.github.io/)** — a high-performance Datalog *compiler* (to C++) used heavily in program analysis. A native binary, not a Python package.
- **[Datomic](https://www.datomic.com/) / [Datascript](https://github.com/tonsky/datascript)** — Datalog as the query language of a database (Clojure ecosystem).
- **[RDFox](https://www.oxfordsemantic.tech/)**, **[LogicBlox](https://en.wikipedia.org/wiki/LogicBlox)** — commercial knowledge-graph / analytics engines.

To keep this notebook **runnable anywhere with zero installs**, the cell below implements a tiny but faithful Datalog engine — parser + naive bottom-up fixpoint — in ~40 lines of the standard library. Implementing it *is* the refresher: you can see the fixpoint loop that every production engine optimizes. The last example also shows the **pyDatalog** call shape, gated behind an `os.getenv` check so it runs only if you opt in.

In [1]:
import re

# ---- A minimal Datalog engine: parse, evaluate to a least fixpoint, query ----

def is_var(t):
    "Datalog convention: Uppercase-initial token is a variable, else a constant."
    return t[:1].isupper()

def parse_atom(s):
    pred, inner = re.match(r"\s*(\w+)\s*\(([^)]*)\)\s*$", s).groups()
    inner = inner.strip()
    args = tuple(a.strip() for a in inner.split(",")) if inner else ()
    return (pred, args)

def parse_rule(line):
    line = line.strip().rstrip(".")
    if ":-" in line:                                   # Head :- Body
        head_s, body_s = line.split(":-", 1)
        body = [parse_atom(a) for a in re.findall(r"\w+\([^)]*\)", body_s)]
        return (parse_atom(head_s), body)
    return (parse_atom(line), [])                      # a fact (empty body)

def unify(pat, fact, subst):
    "Match a (possibly non-ground) atom against a ground fact, extending subst."
    if pat[0] != fact[0] or len(pat[1]) != len(fact[1]):
        return None
    s = dict(subst)
    for p, c in zip(pat[1], fact[1]):
        if is_var(p):
            if s.get(p, c) != c:
                return None
            s[p] = c
        elif p != c:
            return None
    return s

def match_body(body, facts, s):
    "Yield every substitution that satisfies all body atoms (a relational join)."
    if not body:
        yield s
        return
    for f in facts:
        s2 = unify(body[0], f, s)
        if s2 is not None:
            yield from match_body(body[1:], facts, s2)

def ground(atom, s):
    return (atom[0], tuple(s.get(t, t) for t in atom[1]))

def solve(program):
    "Naive bottom-up evaluation: fire all rules until the fact set stops growing."
    rules = [parse_rule(l) for l in program.strip().splitlines() if l.strip()]
    facts = {h for h, b in rules if not b}             # EDB
    rules = [(h, b) for h, b in rules if b]            # IDB rules
    rounds = 0
    while True:
        rounds += 1
        derived = {ground(h, s) for h, b in rules for s in match_body(b, facts, {})}
        new = derived - facts
        if not new:
            break
        facts |= new
    return facts, rounds

def query(facts, pattern):
    "Return the sorted set of tuples matching a query atom, e.g. 'ancestor(tom, Y)'."
    pat = parse_atom(pattern)
    return sorted({f[1] for f in facts if unify(pat, f, {}) is not None})

print("engine ready — stdlib only, no installs")

engine ready — stdlib only, no installs


## 5. Worked Examples

### Example 1 — Recursion: transitive closure (the thing SQL can't do cleanly)

The canonical Datalog program: `ancestor` is the transitive closure of `parent`. The base rule seeds it from direct parents; the recursive rule extends one hop at a time. Bottom-up evaluation keeps firing both rules until no new ancestor pair appears.

In [2]:
family = """
parent(tom, bob).
parent(bob, ann).
parent(bob, pat).
parent(pat, jim).

ancestor(X, Y) :- parent(X, Y).
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).
"""

facts, rounds = solve(family)

print(f"reached fixpoint in {rounds} rounds")
print("everyone tom is an ancestor of :", [y for (_, y) in query(facts, "ancestor(tom, Y)")])
print("all of jim's ancestors        :", [x for (x, _) in query(facts, "ancestor(X, jim)")])
print("total ancestor facts derived  :", sum(f[0] == "ancestor" for f in facts))

reached fixpoint in 4 rounds
everyone tom is an ancestor of : ['ann', 'bob', 'jim', 'pat']
all of jim's ancestors        : ['bob', 'pat', 'tom']
total ancestor facts derived  : 8


### Example 2 — Termination on a cyclic graph (where Prolog would loop)

Graph reachability with the *same* two-rule shape. The graph contains a **cycle** `a → b → c → a`. A naive top-down Prolog search with this left/right-recursive rule can spin forever; Datalog's bottom-up fixpoint simply notices that no *new* `reach` facts appear and stops. This is the decidability guarantee in action — and note the result is a **set** (each reachable pair once, no duplicates from the multiple paths).

In [3]:
graph = """
edge(a, b).
edge(b, c).
edge(c, a).
edge(c, d).

reach(X, Y) :- edge(X, Y).
reach(X, Y) :- edge(X, Z), reach(Z, Y).
"""

facts, rounds = solve(graph)

print(f"terminated after {rounds} rounds on a CYCLIC graph")
print("reachable from a :", [y for (_, y) in query(facts, "reach(a, Y)")])
print("reachable from d :", [y for (_, y) in query(facts, "reach(d, Y)")] or "(nothing)")
print("can a reach itself? ", ("a",) in [(y,) for (_, y) in query(facts, "reach(a, Y)")])

terminated after 4 rounds on a CYCLIC graph
reachable from a : ['a', 'b', 'c', 'd']
reachable from d : (nothing)
can a reach itself?  True


### Example 3 — The real-tooling call shape (pyDatalog), gated so the notebook always runs

In practice you'd reach for a maintained engine. Here's the same ancestor program in **pyDatalog**. The cell only executes the library path if you set `RUN_PYDATALOG=1` *and* the package is installed — otherwise it just prints the code, so the notebook still runs top-to-bottom in a fresh kernel with no extra dependencies.

In [4]:
import os

PYDATALOG_SNIPPET = """
from pyDatalog import pyDatalog
pyDatalog.create_terms("parent, ancestor, X, Y, Z")

+ parent("tom", "bob")
+ parent("bob", "ann")
+ parent("bob", "pat")

ancestor(X, Y) <= parent(X, Y)
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)

print(ancestor("tom", Y))   # -> bindings for Y: bob, ann, pat
"""

if os.getenv("RUN_PYDATALOG") == "1":
    try:
        exec(PYDATALOG_SNIPPET)
    except ImportError:
        print("pyDatalog not installed — run `pip install pyDatalog` first.")
else:
    print("[skipped] set RUN_PYDATALOG=1 (and `pip install pyDatalog`) to execute.")
    print("Call shape for reference:")
    print(PYDATALOG_SNIPPET)

[skipped] set RUN_PYDATALOG=1 (and `pip install pyDatalog`) to execute.
Call shape for reference:

from pyDatalog import pyDatalog
pyDatalog.create_terms("parent, ancestor, X, Y, Z")

+ parent("tom", "bob")
+ parent("bob", "ann")
+ parent("bob", "pat")

ancestor(X, Y) <= parent(X, Y)
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)

print(ancestor("tom", Y))   # -> bindings for Y: bob, ann, pat



## 6. Gotchas & Pitfalls

- **No function symbols — on purpose.** You can't build terms like `succ(succ(0))` or lists. That restriction is *exactly* what makes the Herbrand universe finite and evaluation guaranteed to terminate. If you need structured terms, you want Prolog, not Datalog.
- **Safety / range restriction.** Every head variable must appear in a positive body atom. `bigger(X, Y) :- X > Y.` is unsafe — `X` and `Y` range over infinitely many values. Bind them first: `bigger(X, Y) :- num(X), num(Y), X > Y.`
- **Negation must be stratified.** `p(X) :- not p(X).` has no sensible meaning. Engines require predicates to be layered so no predicate depends (through a cycle) on its own negation; with negation, evaluate stratum by stratum, lowest first.
- **Negation is closed-world.** `not reachable(a, z)` means "we *could not derive* it," not "it's false in some absolute sense." Same closed-world assumption as Prolog's `\+`.
- **Naive evaluation is quadratic-ish in re-work.** The simple engine above re-derives every fact every round. Production engines use **semi-naive** evaluation (join only against last round's *new* facts) — same fixpoint, dramatically less recomputation. Know the difference exists even if you rarely implement it.
- **Set semantics, not bag semantics.** Datalog yields a *set* of facts: no duplicates, no inherent order, no "count how many paths." Pure Datalog can't aggregate; counting/summing needs an aggregation extension (most real engines add one).
- **Rule order doesn't matter (unlike Prolog).** Bottom-up evaluation reaches the same least fixpoint regardless of the order you write clauses. Reordering can't change results or cause non-termination.
- **Recursion through negation or aggregation breaks the guarantees.** Recursive + negated/aggregated rules need extra semantics (well-founded / stable models — see [[answer-set-programming]]) and aren't plain Datalog.

## 7. When to Use vs Alternatives

| Option | Sweet spot | Trade-off vs Datalog |
|--------|-----------|----------------------|
| **Datalog** | Recursive queries over relational/graph data; deductive DBs; program analysis; policy & KG reasoning | Decidable & always terminates, declarative, order-independent — but **no function symbols, limited negation, no native aggregation**. |
| **SQL (`WITH RECURSIVE`)** | Set-based queries over tables; mature engines, indexes, transactions | Can express recursion but verbosely and often inefficiently; Datalog is far cleaner for transitive/mutually-recursive rules. |
| **[[swi-prolog]] (Prolog)** | General logic programming with terms, lists, I/O, search | More expressive (function symbols, cut, DCGs) but **top-down search can loop**; termination is your responsibility. Datalog = Prolog's decidable core. |
| **[[answer-set-programming]] (ASP / clingo)** | Combinatorial search, planning, "choose a model", recursion *through* negation | Stable-model semantics, true non-stratified negation & disjunction; finds all models. Heavier; overkill for plain deductive querying. |
| **[[z3-smt]] (Z3 / SMT)** | Constraints over rich theories (ints, reals, arrays) | A decision procedure, not a recursive query language. Different job. |
| **[[knowledge-graphs]] / [[sparql]]** | Querying RDF triple stores | SPARQL property paths give *some* recursion; Datalog-style engines (RDFox) add full rule-based inference on top of graphs. |
| **Graph databases (Cypher/Gremlin)** | Path queries over property graphs | Imperative-ish traversal DSLs; Datalog is more declarative and composes rules better, but graph DBs win on tooling/ops maturity. |

**Rule of thumb:** if your problem is *recursive queries over relations* and you value a guarantee that every query terminates, Datalog is the precise tool. Need richer terms or general computation → Prolog. Need to *search for models* with unrestricted negation → ASP. Just need non-recursive set queries → plain SQL.

## 8. Resources

- **What You Always Wanted to Know About Datalog (And Never Dared to Ask)** — Ceri, Gottlob & Tanca; the classic survey: <https://www.cs.utexas.edu/~gabriela/cs395t-fall08/papers/datalog.pdf>
- **Soufflé — high-performance Datalog** (docs + tutorial; the engine to know for real work): <https://souffle-lang.github.io/tutorial>
- **Stanford "Introduction to Databases" — Datalog notes**: <https://web.stanford.edu/class/cs345d-01/rl/datalog.pdf>
- **pyDatalog** (the embeddable Python engine used above): <https://sites.google.com/site/pydatalog/>
- **Wikipedia: Datalog** (solid overview of semantics, safety, stratification): <https://en.wikipedia.org/wiki/Datalog>
- **Datomic's Datalog query reference** (Datalog as a practical DB query language): <https://docs.datomic.com/query/query-data-reference.html>